#### 1. Метрики поиска (Оцениваем FAISS / Векторную базу)
Здесь работают строгие математические метрики ранжирования. Для оценки тебе понадобится тестовый датасет (золотой стандарт): 30–50 пар «Вопрос клиента 
→ ID правильного документа».

Precision@K (Точность в топ-K): Доля релевантных документов среди K найденных. Если FAISS вернул 3 документа, и только 1 из них правильный, Precision@3 = 33%.

Recall@K (Полнота в топ-K): Нашел ли алгоритм вообще нужный документ в топ-K выдачи (1 или 0). На хакатонах чаще всего смотрят именно на Recall@3. Если нужный тариф попал в тройку лидеров поиска — поиск отработал успешно, дальше с текстом разберется LLM.

MRR (Mean Reciprocal Rank): Учитывает позицию правильного ответа в выдаче. Если правильный тариф на 1-м месте, оценка 1. Если на 2-м — 0.5. Если на 3-м — 0.33.

#### 2. Классические метрики текста (Устаревший подход)
Раньше качество генерации измеряли путем прямого побуквенного сравнения ответа модели с эталонным ответом человека.

BLEU (Bilingual Evaluation Understudy): Считает точность — сколько слов (n-грамм) из сгенерированного текста совпало с эталоном. Метрика пришла из машинного перевода.

ROUGE (Recall-Oriented Understudy for Gisting Evaluation): Считает полноту — какая доля слов из эталонного текста попала в генерацию. Традиционно применяется для задач суммаризации.

Почему их нельзя нести на хакатон: Если эталон — "Тариф МТС стоит 500 рублей", а LLM ответила "Стоимость этого тарифа составляет 500 руб", BLEU и ROUGE покажут качество около 10–15%, хотя по смыслу ответ идеален. Жюри сразу поймет, что ты применяешь устаревшие методы к современным генеративным сетям.

#### 3. Современный стандарт: LLM-as-a-Judge (Триада RAG)
Индустриальный стандарт оценки RAG — использование мощной LLM (например, GPT-4, Claude или Qwen-Max) в качестве автоматического судьи. Оценка бьется на три вектора (RAG Triad):

Context Relevance (Релевантность контекста): Судья смотрит на «Вопрос клиента» и «Найденный FAISS документ». Оценивает (обычно бинарно 1/0), действительно ли в документе есть информация для ответа. Метрика выявляет мусор в поисковой выдаче.

Groundedness / Faithfulness (Фактическая точность / Отсутствие галлюцинаций): Судья сравнивает «Найденный контекст» и «Ответ твоей LLM». Выявляет, не придумала ли твоя модель цены или услуги, которых не было в документации. Это главная метрика безопасности для B2B-продуктов.

Answer Relevance (Полезность ответа): Судья сравнивает «Вопрос клиента» и «Финальный ответ». Проверяет, ответила ли модель на суть вопроса, или просто слепо пересказала найденный контекст.

Как презентовать это жюри на хакатоне
Собери Evaluation Set: Напиши руками 30 каверзных вопросов по тарифам из базы.

Собери логи: Прогони вопросы через свой RAG-алгоритм и сохрани результаты в таблицу (Вопрос 
→ Найденный документ 
→ Сгенерированный ответ).

Запусти авто-судью: Напиши промпт для тяжелой LLM по API (или попроси любую доступную сильную модель), чтобы она проставила 1 или 0 для метрик Groundedness и Answer Relevance для каждой строки.

Выведи на слайд: "Метрика Groundedness = 98% (Бот не галлюцинирует и не придумывает тарифы). Recall@3 векторного поиска = 94%."